# 03 - Feature engineering sans fuite (Phase 4)
**Objectif :** construire (a) la table client pour la segmentation, (b) la table `client × snapshot` pour le churn, **sans aucune fuite temporelle**.

Principe : à chaque date d'observation **T**, les variables n'utilisent que le passé (`InvoiceDate < T`) ; la cible regarde le futur (`T ≤ InvoiceDate < T + horizon`).

In [ ]:
import sys, warnings
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.config import load_config, resolve, save_table, load_table
cfg = load_config()
FIG = resolve(cfg, "figures_dir")
OUT = resolve(cfg, "outputs_dir")
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 50)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True, "grid.alpha": .3})

from src.features import (build_features, build_target, build_churn_dataset, build_customer_features,
                          snapshot_schedule, check_split_purge)
from src.categories import export_validation_sample
df = load_table("transactions_clean", cfg)
HORIZON = cfg["churn"]["horizon_days"]
print(f"Données : {df['InvoiceDate'].min():%Y-%m-%d} -> {df['InvoiceDate'].max():%Y-%m-%d} | horizon = {HORIZON} j")

## 1. Calendrier des snapshots et règle de purge
Une fenêtre de cible `[T, T+90j)` regarde le futur. Si la cible d'un snapshot d'entraînement se termine **après** le début de la validation, le modèle a « vu » une partie de l'avenir de la validation (fuite). On impose donc que la cible d'un split finisse **avant** le premier T du split suivant.

In [ ]:
sched = snapshot_schedule(cfg, df, HORIZON)
print(sched.to_string(index=False))
problems = check_split_purge(sched)
print("\nProblèmes de fuite :", problems or "aucun")
assert sched["fenetre_complete"].all(), "un snapshot dépasse la fin des données"

fig, ax = plt.subplots(figsize=(11, 4)); colors = {"train": "#2b6cb0", "validation": "#d69e2e", "test": "#c53030"}
for i, r in sched.iterrows():
    ax.barh(i, (r["label_end"] - r["T"]).days, left=r["T"], color=colors[r["split"]], alpha=.8)
    ax.plot(r["T"], i, "k|", ms=12)
ax.set_yticks(range(len(sched))); ax.set_yticklabels([f"T={t:%Y-%m-%d} ({s})" for t, s in zip(sched["T"], sched["split"])])
ax.set_title("Fenêtres de cible [T, T+90 j) par snapshot (bleu = train, jaune = validation, rouge = test)")
plt.tight_layout(); plt.savefig(FIG / "calendrier_snapshots.png", dpi=150); plt.show()

## 2. Exemple : features d'un snapshot et preuve d'absence de fuite

In [ ]:
T = pd.Timestamp("2011-03-31")
f = build_features(df, T, cfg)
print(f"{len(f):,} clients avec au moins un achat avant {T.date()}")
f.head()

In [ ]:
# Preuve : on supprime TOUT ce qui est >= T ; les features doivent être strictement identiques.
f_past_only = build_features(df[df["InvoiceDate"] < T], T, cfg)
pd.testing.assert_frame_equal(f.sort_values("CustomerID").reset_index(drop=True),
                              f_past_only.sort_values("CustomerID").reset_index(drop=True))
print("OK : les features ne dépendent d'aucune donnée postérieure à T")

## 3. Dictionnaire des variables

In [ ]:
desc = {
 "recency": "jours entre le dernier achat et T",
 "frequency": "nombre de factures de vente avant T (les retours ne comptent pas)",
 "monetary": "CA net = ventes - retours",
 "gross_revenue": "CA des ventes seules",
 "return_rate": "retours / ventes (borné à [0, 1])",
 "tenure_days": "ancienneté : jours depuis le premier achat",
 "avg_basket": "panier moyen (CA par facture)",
 "avg_items_per_invoice": "articles par facture",
 "avg_qty_per_line": "quantité moyenne par ligne (sert à détecter les grossistes)",
 "n_unique_products / n_unique_categories": "diversité des produits / catégories achetés",
 "ipt_mean": "intervalle moyen entre deux jours d'achat (NaN si un seul achat)",
 "is_single_purchase": "1 si un seul jour d'achat",
 "recency_over_ipt": "récence / intervalle habituel (>1 : en retard sur son rythme)",
 "revenue_recent / revenue_prev": "CA des 90 derniers jours / des 90 jours précédents",
 "trend": "(recent - prev) / (|recent| + |prev|) dans [-1, 1]",
 "is_wholesale": "1 si quantité moyenne par ligne >= seuil fixe (heuristique B2B)",
 "country_group / is_uk": "UK / Europe / Rest of World ; indicateur UK",
 "cat_share_*": "part du CA par catégorie dérivée de la Description (approximative)",
 "snapshot_month": "mois de T (saisonnalité)",
}
pd.DataFrame({"variable": desc.keys(), "définition": desc.values()})

## 4. Valeurs manquantes à traiter à la phase de modélisation

In [ ]:
na = f.isna().mean().mul(100).round(1); na[na > 0]

`ipt_mean` et `recency_over_ipt` sont manquants pour les clients à achat unique : c'est **normal** (information réelle, signalée par `is_single_purchase`). L'imputation se fera **dans le pipeline de modélisation, ajustée sur le train uniquement**.

## 5. Catégories dérivées de la Description : validation sur échantillon étiqueté à la main

In [ ]:
print((100 * df.drop_duplicates("StockCode")["Category"].value_counts(normalize=True)).round(1))
sample = export_validation_sample(df, n=150, seed=cfg["seed"])
path = OUT / "validation_categories_sample.csv"
if not path.exists():
    sample.to_csv(path, index=False)
    print("\nÉchantillon écrit dans", path, "\n-> remplis la colonne 'Category_manual' à la main (valeurs : voir CATEGORY_KEYWORDS ou OTHER), puis relance la cellule suivante.")
sample.head(10)

In [ ]:
lab = pd.read_csv(path)
lab = lab[lab["Category_manual"].notna() & (lab["Category_manual"].astype(str).str.strip() != "")]
if len(lab) >= 30:
    acc = (lab["Category_auto"].str.upper() == lab["Category_manual"].astype(str).str.upper().str.strip()).mean()
    print(f"Précision de la catégorisation automatique sur {len(lab)} produits étiquetés : {100*acc:.1f} %")
    print(pd.crosstab(lab["Category_manual"].str.upper(), lab["Category_auto"]))
else:
    print(f"{len(lab)} produits étiquetés : étiquette au moins 100 produits pour mesurer la précision.")

## 6. Table pour la segmentation (`customer_features`)
Features sur tout l'historique, T = fin des données + 1 jour. Cette table est **descriptive** (pas de cible).

In [ ]:
seg = build_customer_features(df, cfg)
print(f"{len(seg):,} clients | T = {seg['snapshot_date'].iloc[0]:%Y-%m-%d}")
print(save_table(seg, "customer_features", cfg))
seg[["recency", "frequency", "monetary", "avg_basket", "tenure_days", "return_rate"]].describe().round(1)

## 7. Table pour le churn (`churn_dataset` : client × snapshot)

In [ ]:
churn = build_churn_dataset(df, cfg, HORIZON)
print(save_table(churn, "churn_dataset", cfg))
tab = churn.groupby(["split", "snapshot_date"])["churn"].agg(lignes="size", taux_churn="mean").round(3)
tab

In [ ]:
fig, ax = plt.subplots(); churn.groupby("snapshot_date")["churn"].mean().plot(marker="o", ax=ax)
ax.set_ylabel("taux de churn"); ax.set_title(f"Taux de churn par snapshot (horizon {HORIZON} j)")
plt.savefig(FIG / "taux_churn_snapshots.png", dpi=150); plt.show()
print("Un taux de churn stable d'un snapshot à l'autre est bon signe ; une forte variation est à expliquer (saisonnalité).")

## 8. Analyse de sensibilité de l'horizon (60 / 90 / 120 jours)

In [ ]:
rows = []
for h in cfg["churn"]["sensitivity_horizons"]:
    for _, r in snapshot_schedule(cfg, df, h).iterrows():
        if not r["fenetre_complete"]:
            continue
        feats = build_features(df, r["T"], cfg)
        t = build_target(df, r["T"], h, feats["CustomerID"])
        rows.append({"horizon": h, "T": r["T"].date(), "split": r["split"], "taux_churn": t["churn"].mean()})
sens = pd.DataFrame(rows)
sens.pivot_table(index=["T", "split"], columns="horizon", values="taux_churn").round(3)

*Remarque :* pour l'horizon 120 j, certains snapshots tardifs sont exclus (fenêtre incomplète) et le contrôle de purge de `build_churn_dataset` doit être relancé avec ce même horizon avant la phase 6.

## 9. Corrélations entre variables (anticiper la multicolinéarité)

In [ ]:
num = ["recency", "frequency", "monetary", "avg_basket", "tenure_days", "return_rate", "n_unique_products", "ipt_mean", "trend"]
corr = np.log1p(seg[num].clip(lower=0)).corr(method="spearman").round(2)
fig, ax = plt.subplots(figsize=(7, 6)); im = ax.imshow(corr, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(num))); ax.set_xticklabels(num, rotation=60, ha="right"); ax.set_yticks(range(len(num))); ax.set_yticklabels(num)
for i in range(len(num)):
    for j in range(len(num)): ax.text(j, i, corr.iloc[i, j], ha="center", va="center", fontsize=7)
plt.colorbar(im); plt.title("Corrélation de Spearman (log1p)"); plt.tight_layout(); plt.savefig(FIG / "correlations.png", dpi=150); plt.show()

## 10. Livrables de la phase 4
- `data/processed/customer_features` (segmentation)
- `data/processed/churn_dataset` (churn, colonne `split` = train / validation / test)
- `tests/test_features_no_leakage.py` : `pytest tests -q` doit afficher « passed »

**Rappel :** si les segments K-Means sont utilisés plus tard comme variable du modèle de churn, ils doivent être recalculés à chaque snapshot (sinon fuite).